# Member 4: spectrum_formula.arch2_contrastive on Kaggle (train + predict, offline)

Contrastive spectrum-molecule model on the **team protocol** (Member 3's split, pool, ranking helpers and scorer):
spectra and molecules are embedded into one space, trained with in-batch and same-mass hard negatives, and
candidates are ranked by cosine similarity. Optional gate fuses library search where it is reliable.

**Attach before running:** the competition data and your private datasets `casmi-m4-code`, `casmi26-coconut`,
`rdkit-wheel`. Internet off; GPU on. Set `M4_QUICK=1` for a short wiring check.

In [ ]:
import glob
import importlib
import os
import shutil
import subprocess
import sys
import time

import pandas as pd

T0 = time.time()
QUICK = os.environ.get("M4_QUICK", "0") == "1"      # 1 = tiny wiring check

def find(pattern):
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    if not hits:
        raise FileNotFoundError(f"nothing matching {pattern} under /kaggle/input; attach the dataset")
    return hits[0]

CODE = os.path.dirname(find("CASMI_M4_CODE"))          # repo layout: src/casmi, configs, models/, baselines/
CAND_DIR = "/kaggle/working/candidates"   # candidate tables from separate datasets, gathered in one folder
os.makedirs(CAND_DIR, exist_ok=True)
for name in ("coconut.parquet", "pubchemlite.parquet", "pubchemlite_counts.parquet"):
    try:
        shutil.copy(find(name), CAND_DIR)
    except FileNotFoundError:
        print(f"{name} not attached")
sys.path[:0] = [f"{CODE}/src", f"{CODE}/models/spectrum_formula/arch2_contrastive"]
os.environ.update(CASMI_ENV="kaggle", CASMI_CANDIDATES_DIR=CAND_DIR)
print("code:", CODE, "| candidates:", CAND_DIR, "| python", sys.version.split()[0])

In [ ]:
# RDKit is not in Kaggle's image: install the wheel that matches this Python, offline.
try:
    import rdkit  # noqa: F401
except ImportError:
    tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
    wheel = find(f"rdkit*{tag}*.whl")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--no-deps", wheel], check=True)
    importlib.invalidate_caches()
import rdkit

print("RDKit", rdkit.__version__)   # the metric pins 2026.03.3

In [ ]:
import torch
import yaml

cfg = yaml.safe_load(open(f"{CODE}/models/spectrum_formula/arch2_contrastive/config.yaml"))
cfg["data"].update(external_pools=["coconut"])     # candidate pool: training structures + COCONUT
EXTRA = {}         # per-run data overrides, e.g. {"bin_width": 0.5, "neutral_loss": True}
EXTRA_MODEL = {}   # per-run model overrides, e.g. {"spec_dim": 1024, "hidden": 2048}
EXTRA_TRAIN = {}   # per-run training overrides, e.g. {"epochs": 24}
EXTRA_PREDICT = {}  # per-run predict overrides, e.g. {"prior": {"weight": 0.8, "topk": 25}}
RUN_GATE = True    # fuse library search (B2) where it is reliable (Member 7's gate)
cfg["data"].update(EXTRA)
cfg["model"].update(EXTRA_MODEL)
if QUICK:
    cfg["data"].update(per_structure=1, n_val=400)
    cfg["train"].update(epochs=2)
else:
    cfg["data"].update(per_structure=4, n_val=3000)
    cfg["train"].update(epochs=12)
cfg["train"].update(EXTRA_TRAIN)
cfg["predict"].update(EXTRA_PREDICT)
print(torch.cuda.device_count(), "GPU(s):", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
print(yaml.safe_dump(cfg, sort_keys=False))

In [ ]:
# Team protocol: split, pool (+ COCONUT) and pool fingerprints are built once and reused below.
import train as m4_train

proto = m4_train.build_protocol(cfg)
n_test = proto.test.molecule_id.nunique()
print(f"pool {len(proto.pool):,} structures | val molecules {len(proto.truth):,} | test molecules {n_test:,}")

In [ ]:
ckpt = m4_train.train(cfg, proto)

In [ ]:
# Validation on the team protocol, then the test submission (optionally gated with library search).
import predict as m4_predict
from common import b2gate

cp = proto.cp
pool_keys = proto.pool.key.to_numpy()
val_rank, _, truth, _ = m4_predict.predict(cfg, "val", proto)
test_rank, _, _, _ = m4_predict.predict(cfg, "test", proto)
tables = {"model": cp.evaluate(val_rank, truth, pool_keys)}
honest = {"model": float(tables["model"].loc["balanced", "MRR@25"])}
final_rank, chosen, gate_info = test_rank, "model", None
print("model only:\n", tables["model"].to_string())

if RUN_GATE:
    val_g, test_g, gate_info, _ = b2gate.fuse_with_b2(proto, val_rank, test_rank, log=print)
    tables["gated"] = cp.evaluate(val_g, truth, pool_keys)
    honest["gated"] = gate_info["cross_fitted"]          # tau was tuned on validation: judge it out of sample
    print("gate:", gate_info, "\ngated:\n", tables["gated"].to_string())
    if honest["gated"] > honest["model"]:
        final_rank, chosen = test_g, "gated"

cp.write_submission(test_rank, proto.pool, proto.test, path="/kaggle/working/submission_model_only.csv")
sub = cp.write_submission(final_rank, proto.pool, proto.test, path="/kaggle/working/submission.csv")
pd.concat(tables, names=["method"]).to_csv("/kaggle/working/validation_table.csv")
print(f"submission uses: {chosen} | honest val {honest} | {sub.shape} | total {(time.time() - T0) / 60:.1f} min")
sub.head()